# Predicting below-average US nursing homes

MLDA CW1, 00017694. CMS Care Compare, Oct 2026.

## A. Data load

4 files, joined on CCN.

In [ ]:
import pandas as pd
import numpy as np

### A.1 Loading the files

I read CCN as string, otherwise 015009 becomes 15009.

In [ ]:
# key column, same in all 4 files
CCN = 'CMS Certification Number (CCN)'

provider = pd.read_csv('../data/raw/NH_ProviderInfo_Oct2026.csv', dtype={CCN: str})
citations = pd.read_csv('../data/raw/NH_HealthCitations_Oct2026.csv', dtype={CCN: str})
penalties = pd.read_csv('../data/raw/NH_Penalties_Oct2026.csv', dtype={CCN: str})
ownership = pd.read_csv('../data/raw/NH_Ownership_Oct2026.csv', dtype={CCN: str})

print(provider.shape, citations.shape, penalties.shape, ownership.shape)

In [ ]:
provider.head()

Checking there are no duplicate homes.

In [ ]:
print(len(provider), provider[CCN].nunique())

### A.2 Citations

Many rows per home, so I count them per home with groupby.

In [ ]:
citations['cit_total'] = 1
citations['cit_complaint'] = (citations['Complaint Deficiency'] == 'Y').astype(int)

# G-L = residents were harmed
citations['cit_harm'] = citations['Scope Severity Code'].isin(['G', 'H', 'I', 'J', 'K', 'L']).astype(int)

cit_per_home = citations.groupby(CCN)[['cit_total', 'cit_complaint', 'cit_harm']].sum().reset_index()
cit_per_home.head()

### A.3 Penalties

In [ ]:
penalties['pen_fines'] = (penalties['Penalty Type'] == 'Fine').astype(int)
penalties['pen_fines_usd'] = penalties['Fine Amount'].fillna(0)

pen_per_home = penalties.groupby(CCN)[['pen_fines', 'pen_fines_usd']].sum().reset_index()
pen_per_home.head()

### A.4 Ownership

This file has managers too, not only owners.

In [ ]:
ownership['Role played by Owner or Manager in Facility'].value_counts()

In [ ]:
owner_roles = ['5% OR GREATER DIRECT OWNERSHIP INTEREST', '5% OR GREATER INDIRECT OWNERSHIP INTEREST',
               'DIRECT OWNERSHIP INTEREST', 'INDIRECT OWNERSHIP INTEREST']

ownership['own_owners'] = ownership['Role played by Owner or Manager in Facility'].isin(owner_roles).astype(int)
ownership['own_org_owners'] = ownership['own_owners'] * (ownership['Owner Type'] == 'Organization')

own_per_home = ownership.groupby(CCN)[['own_owners', 'own_org_owners']].sum().reset_index()
own_per_home.head()

### A.5 Merging

I use left join so no home is lost.

In [ ]:
df = provider.merge(cit_per_home, on=CCN, how='left')
df = df.merge(pen_per_home, on=CCN, how='left')
df = df.merge(own_per_home, on=CCN, how='left')

print(df.shape)

Homes not in a file had nothing, so NaN means 0.

In [ ]:
new_cols = ['cit_total', 'cit_complaint', 'cit_harm', 'pen_fines', 'pen_fines_usd', 'own_owners', 'own_org_owners']

print(df[new_cols].isna().sum())
df[new_cols] = df[new_cols].fillna(0)

Checking my fines count against the CMS column.

In [ ]:
print((df['Number of Fines'] == df['pen_fines']).mean())

Note: cit_ and pen_ come from the inspections used for the rating, so I won't use them as features (leakage).

In [ ]:
df.to_csv('../data/processed/nursing_homes_joined.csv', index=False)